In [1]:
# 直接导入我们在终端已经安装好的 PyG
import torch
import torch_geometric
import torch_geometric.nn as geom_nn
import torch_geometric.data as geom_data

# 打印一下版本，确保安装成功且 Kernel 已经识别
print(f"PyTorch Geometric version: {torch_geometric.__version__}")

PyTorch Geometric version: 2.6.1


In [5]:
## Standard libraries
import os
import json
import math
import numpy as np 
import time

## Imports for plotting
import matplotlib.pyplot as plt
%matplotlib inline 
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('svg', 'pdf') # For export
from matplotlib.colors import to_rgb
import matplotlib
matplotlib.rcParams['lines.linewidth'] = 2.0
import seaborn as sns
sns.reset_orig()
sns.set()

## Progress bar
from tqdm.notebook import tqdm

## PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torch.optim as optim
# Torchvision
import torchvision
from torchvision.datasets import CIFAR10
from torchvision import transforms
# PyTorch Lightning
try:
    import pytorch_lightning as pl
except ModuleNotFoundError: # Google Colab does not have PyTorch Lightning installed by default. Hence, we do it here if necessary
    !pip install --quiet pytorch-lightning>=1.4
    import pytorch_lightning as pl
from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint

# Path to the folder where the datasets are/should be downloaded (e.g. CIFAR10)
DATASET_PATH = "../data"
# Path to the folder where the pretrained models are saved
CHECKPOINT_PATH = "../saved_models/tutorial7"

# Setting the seed
pl.seed_everything(42)

# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)

C:\Users\loktsinglong\AppData\Local\Temp\ipykernel_46752\1415272395.py:12: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats('svg', 'pdf') # For export
Seed set to 42


cpu


In [2]:
gnn_layer_by_name = {
    "GCN":geom_nn.GCNConv,
    "GAT":geom_nn.GATConv,
    "GraphConv":geom_nn.GraphConv
}

In [3]:
import os

# 1. 定义本地存放数据的文件夹名称（用相对路径，保存在你当前代码同级的 data 文件夹下）
DATASET_PATH = "./data"

# 2. 如果文件夹不存在，Python 会自动帮你建一个
os.makedirs(DATASET_PATH, exist_ok=True)

# 3. 下载并加载 Cora 数据集
cora_dataset = torch_geometric.datasets.Planetoid(root=DATASET_PATH, name="Cora")

print("数据集下载/加载成功！")

Processing...


数据集下载/加载成功！


Done!


In [4]:
cora_dataset[0] # Cora dataset only has one graph,so we can directly access it with index 0.

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])

In [12]:
class GNNModel(nn.Module):
    def __init__(self,c_in,c_hidden,c_out,num_layers=2,layer_name="GCN",dp_rate=0.1,**kwargs):
        """
        Inputs:
            c_in - Dimension of input features
            c_hidden - Dimension of hidden features
            c_out - Dimension of the output features. Usually number of classes in classification
            num_layers - Number of "hidden" graph layers
            layer_name - String of the graph layer to use
            dp_rate - Dropout rate to apply throughout the network
            kwargs - Additional arguments for the graph layer (e.g. number of heads for GAT)
        """
        super().__init__()
        gnn_layer = gnn_layer_by_name[layer_name]

        layers = []
        in_channels, out_channels = c_in, c_hidden
        for l_idx in range(num_layers-1):
            layers += [
                gnn_layer(in_channels=in_channels,
                         out_channels=out_channels,
                         **kwargs),
                nn.ReLU(inplace=True),
                nn.Dropout(dp_rate)
            ]
            in_channels = c_hidden
        layers += [gnn_layer(in_channels=in_channels,
                         out_channels=c_out,
                         **kwargs)]
        self.layers = nn.ModuleList(layers)
    
    def forward(self,x,edge_index):
        """
        Inputs:
            x - Input features per node
            edge_index - List of vertex index pairs representing the edges in the graph (PyTorch geometric notation)
        """
        for l in self.layers:
            # For graph layers, we need to add the "edge_index" tensor as additional input
            # All PyTorch Geometric graph layer inherit the class "MessagePassing", hence
            # we can simply check the class type.
            if isinstance(l,geom_nn.MessagePassing):
                x=l(x,edge_index)
            else:
                x=l(x)
        return x
        

In [ ]:
class MLPModule(nn.Module):
    def __init__(self,c_in,c_hidden,c_out,num_layers=2,dp_rate=0.1):
        """
        Inputs:
            c_in - Dimension of input features
            c_hidden - Dimension of hidden features
            c_out - Dimension of the output features. Usually number of classes in classification
            num_layers - Number of hidden layers
            dp_rate - Dropout rate to apply throughout the network
        """
        super().__init__()
        layers= []
        in_channels,out_channels = c_in, c_hidden
        for l_idx in range(num_layers-1):
            layers += [
                nn.Linear(in_channels,out_channels),
                nn.ReLU(inplace=True),
                nn.Dropout(dp_rate)
            ]
            in_channels = c_hidden
            layers += [nn.Linear(in_channels,c_out)]
            self.layers = nn.Sequential(*layers)
    
    def forward(self,x,*args,**kwargs):
        #定义函数加星号（def foo(**kwargs)），是把零散的参数“打包”成字典；* args 是把零散的参数“打包”成元组；而** kwargs 是把零散的参数“打包”成字典。
        #调用函数加星号（foo(**kwargs)），是把字典“解包”成零散的参数！
        # We add *args and **kwargs to the forward function to ensure that it can be called with the same arguments as the GNN model, even though it does not use them.
        """
        Inputs:
            x - Input features per node
        """
        return self.layers(x)

In [21]:
class NodeLevelGNN(pl.LightningModule):

    def __init__(self,model_name,**model_kwargs):
        super().__init__()
        #save hyperparameters
        self.save_hyperparameters()

        if model_name == "MLP":
            self.model = MLPModule(**model_kwargs)
        else:
            self.model = GNNModel(**model_kwargs)
        self.loss_module = nn.CrossEntropyLoss() #多分类交叉熵损失函数

    def forward(self,data,mode="train"):
        x, edge_index = data.x, data.edge_index
        x = self.model(x,edge_index)

        #Only calculate the loss on the nodes corresponding to the mask
        if mode == "train":
            mask = data.train_mask
        elif mode =="val":
            mask = data.val_mask
        elif mode == "test":
            mask = data.test_mask
        else:
            assert False,f"Unknown forward mode:{mode}"
        
        loss = self.loss_module(x[mask],data.y[mask])
        acc = (x[mask].argmax(dim=-1)==data.y[mask]).sum().float()/mask.sum()
        return loss, acc

    def configure_optimizers(self):
        # We use SGD here, but Adam works as well 
        optimizer = optim.SGD(self.parameters(),lr=0.01,momentum=0.9,weight_decay=2e-3)
        return optimizer
    
    def training_step(self,batch,batch_idx):
        loss,acc = self.forward(batch,mode="train")
        self.log("train_loss",loss)
        self.log("train_acc",acc)
        return loss
    
    def validation_step(self,batch,batch_idx):
        _, acc = self.forward(batch,mode="val")
        self.log("val_acc",acc)

    def test_step(self,batch,batch_idx):
        _, acc = self.forward(batch,mode="test")
        self.log("test_acc",acc)

In [19]:
# 【极其重要：请确保在代码文件的最上方有以下导入和全局变量】
# import os
# import torch
# import pytorch_lightning as pl
# from pytorch_lightning.callbacks import ModelCheckpoint
from torch_geometric.loader import DataLoader  #<-- 【关键修改1】新版 PyG 已经把 DataLoader 移到了这里！
# 
# CHECKPOINT_PATH = "./saved_models"
# device = torch.device("cpu") # 你的环境是 CPU 版

def train_node_classifier(model_name, dataset, **model_kwargs):
    # 固定随机种子，确保你每次跑出来的结果都是一样的，方便复现和找 Bug
    pl.seed_everything(42) 
    
    # 【关键修改2】替换掉 geom_data.DataLoader，使用新版的 DataLoader
    
    # 图数据通常只有一张大图，所以 batch_size 设为 1
    node_data_loader = DataLoader(dataset, batch_size=1) 
    
    # 建立一个专门存放当前模型权重的文件夹
    root_dir = os.path.join(CHECKPOINT_PATH, "NodeLevel" + model_name)
    os.makedirs(root_dir, exist_ok=True)
    
    # ==========================================
    # 核心组件：实例化 PyTorch Lightning 全自动训练器
    # ==========================================
    trainer = pl.Trainer(
        default_root_dir=root_dir,
        # 回调函数：时刻盯着验证集准确率 (val_acc)，自动把表现最好的一次权重存下来
        callbacks=[ModelCheckpoint(save_weights_only=True, mode="max", monitor="val_acc")],
        accelerator="gpu" if str(device).startswith("cuda") else "cpu",
        devices=1,
        max_epochs=200, # 总共训练 200 轮
        enable_progress_bar=False # 设为 True 会显示进度条，False 则让控制台保持清爽
    ) 
    trainer.logger._default_hp_metric = None 

    # ==========================================
    # 智能检查：有没有“偷懒”的捷径？
    # ==========================================
    pretrained_filename = os.path.join(CHECKPOINT_PATH, f"NodeLevel{model_name}.ckpt")
    if os.path.isfile(pretrained_filename):
        # 如果发现本地已经有训练好的模型，直接“白嫖”加载，跳过漫长的训练过程！
        print("Found pretrained model, loading...")
        model = NodeLevelGNN.load_from_checkpoint(pretrained_filename)
    else:
        # 如果没有，那就老老实实从零开始建模型
        pl.seed_everything()
        model = NodeLevelGNN(model_name=model_name, c_in=dataset.num_node_features, c_out=dataset.num_classes, **model_kwargs)
        
        # 【起飞指令】把模型和数据丢给 trainer，它会自动帮你把反向传播、梯度更新全做了！
        trainer.fit(model, node_data_loader, node_data_loader)
        
        # 训练结束后，把那 200 轮里准确率最高的那次“黄金权重”加载回模型中
        model = NodeLevelGNN.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
    
    # ==========================================
    # 最终期末考试与成绩单生成
    # ==========================================
    # 拿测试集测试一下最终水平
    test_result = trainer.test(model, node_data_loader, verbose=False)
    
    # 手动提取一次数据，算一下训练集和验证集的最终准确率
    batch = next(iter(node_data_loader))
    batch = batch.to(device) # 【关键修改3】使用全局 device，避免属性找不到的报错
    
    _, train_acc = model.forward(batch, mode="train")
    _, val_acc = model.forward(batch, mode="val")
    
    # 打包成绩单返回
    result = {"train": train_acc,
              "val": val_acc,
              "test": test_result[0]['test_acc']}
    return model, result

In [17]:
# Small function for printing the test scores
def print_results(result_dict):
    if "train" in result_dict:
        print(f"Train accuracy:{(100.0*result_dict['train']):4.2f}%")
    if "val" in result_dict:
        print(f"Val accuracy:{(100.0*result_dict['val']):4.2f}%")
    print(f"Test accuracy:{(100.0*result_dict['test']):4.2f}%")
        


In [24]:
node_mlp_model, node_mlp_result = train_node_classifier(
    model_name="MLP",
    dataset=cora_dataset,
    c_hidden=16,
    num_layers=2,
    dp_rate=0.1
)
print_results(node_mlp_result)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Lightning automatically upgraded your loaded checkpoint from v1.0.2 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint d:\GNN\saved_models\tutorial7\NodeLevelMLP.ckpt`
d:\Anoconda\envs\gnn_env\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Found pretrained model, loading...
Train accuracy:97.86%
Val accuracy:52.80%
Test accuracy:60.60%


d:\Anoconda\envs\gnn_env\lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2708. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


In [25]:
node_gnn_model,node_gnn_result = train_node_classifier(
    model_name="GNN",
    layer_name="GCN",
    dataset=cora_dataset,
    c_hidden=16,
    num_layers=2,
    dp_rate=0.1
)
print_results(node_gnn_result)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Lightning automatically upgraded your loaded checkpoint from v1.0.2 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint d:\GNN\saved_models\tutorial7\NodeLevelGNN.ckpt`


Found pretrained model, loading...
Train accuracy:100.00%
Val accuracy:77.80%
Test accuracy:82.40%


In [26]:
tu_dataset = torch_geometric.datasets.TUDataset(root=DATASET_PATH, name="MUTAG")

Processing...
Done!


In [27]:
print("Data object:", tu_dataset.data)
print("Length:", len(tu_dataset))
print(f"Average label: {tu_dataset.data.y.float().mean().item():4.2f}")


Data object: Data(x=[3371, 7], edge_index=[2, 7442], edge_attr=[7442, 4], y=[188])
Length: 188
Average label: 0.66


d:\Anoconda\envs\gnn_env\lib\site-packages\torch_geometric\data\in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [29]:
torch.manual_seed(42)
tu_dataset.shuffle()
train_dataset = tu_dataset[:150]
test_dataset = tu_dataset[150:]

In [30]:
graph_train_loader = geom_data.DataLoader(train_dataset, batch_size=64, shuffle=True)
graph_val_loader = geom_data.DataLoader(test_dataset, batch_size=64) # Additional loader if you want to change to a larger dataset
graph_test_loader = geom_data.DataLoader(test_dataset, batch_size=64)

d:\Anoconda\envs\gnn_env\lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [31]:

batch = next(iter(graph_test_loader))
print("Batch:", batch)
print("Labels:", batch.y[:10])
print("Batch indices:", batch.batch[:40])

Batch: DataBatch(edge_index=[2, 1512], x=[687, 7], edge_attr=[1512, 4], y=[38], batch=[687], ptr=[39])
Labels: tensor([1, 1, 1, 0, 0, 0, 1, 1, 1, 0])
Batch indices: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2])


In [33]:
class GraphGNNModel(nn.Module):
    def __init__(self,c_in,c_hidden,c_out,dp_rate_linear=0.5,**kwargs):
        """
        Inputs:
            c_in - Dimension of input features
            c_hidden - Dimension of hidden features
            c_out - Dimension of output features (usually number of classes)
            dp_rate_linear - Dropout rate before the linear layer (usually much higher than inside the GNN)
            kwargs - Additional arguments for the GNNModel object
        """
        super().__init__()
        self.GNN = GNNModel(c_in=c_in,
                            c_hidden=c_hidden,
                            c_out=c_hidden,
                            **kwargs)
        self.head = nn.Sequential(
            nn.Dropout(dp_rate_linear),
            nn.Linear(c_hidden,c_out)
        )

    def forward(self,x,edge_index,batch_idx):
        """
        Inputs:
            x - Input features per node
            edge_index - List of vertex index pairs representing the edges in the graph (PyTorch geometric notation)
            batch_idx - Index of batch element for each node
        """
        x = self.GNN(x,edge_index)
        x = geom_nn.global_mean_pool(x,batch_idx) #Average pooling
        x = self.head(x)
        return x
        


In [ ]:
class GraphLevelGNN(pl.LightningModule):
    
    def __init__(self, **model_kwargs):
        super().__init__()
        # Saving hyperparameters
        self.save_hyperparameters()
        
        self.model = GraphGNNModel(**model_kwargs)
        self.loss_module = nn.BCEWithLogitsLoss() if self.hparams.c_out == 1 else nn.CrossEntropyLoss()

    def forward(self, data, mode="train"):
        x, edge_index, batch_idx = data.x, data.edge_index, data.batch
        x = self.model(x, edge_index, batch_idx)
        x = x.squeeze(dim=-1) #[[1.2], [-0.5], [2.1]] --> [1.2, -0.5, 2.1] 
        
        if self.hparams.c_out == 1:
            preds = (x > 0).float() # [1.2, -0.5, 2.1] --> [1, 0, 1]
            data.y = data.y.float() #[1.0, 0.0, 0.0]
        else:
            preds = x.argmax(dim=-1)
        loss = self.loss_module(x, data.y)
        acc = (preds == data.y).sum().float() / preds.shape[0]
        return loss, acc

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-2, weight_decay=0.0) # High lr because of small dataset and small model
        return optimizer

    def training_step(self, batch, batch_idx):
        loss, acc = self.forward(batch, mode="train")
        self.log('train_loss', loss)
        self.log('train_acc', acc)
        return loss

    def validation_step(self, batch, batch_idx):
        _, acc = self.forward(batch, mode="val")
        self.log('val_acc', acc)

    def test_step(self, batch, batch_idx):
        _, acc = self.forward(batch, mode="test")
        self.log('test_acc', acc)

In [35]:
def train_graph_classifier(model_name, dataset, train_loader, val_loader, test_loader, **model_kwargs):
    # 固定随机种子
    pl.seed_everything(42)
    
    # 建好保存权重的文件夹
    root_dir = os.path.join(CHECKPOINT_PATH, "GraphLevel" + model_name)
    os.makedirs(root_dir, exist_ok=True)
    
    # 实例化全自动教练机 Trainer
    trainer = pl.Trainer(
        default_root_dir=root_dir,
        callbacks=[ModelCheckpoint(save_weights_only=True, mode="max", monitor="val_acc")],
        accelerator="cpu", # 【关键修改1】因为你是 CPU 环境，直接写死 cpu 防报错
        devices=1,
        max_epochs=500,    # 图分类通常需要更久的时间收敛，所以给了 500 轮
        enable_progress_bar=True # 开启进度条
    )

    pretrained_filename = os.path.join(CHECKPOINT_PATH, f"GraphLevel{model_name}.ckpt")
    if os.path.isfile(pretrained_filename):
        print("发现预训练模型，正在光速加载...")
        model = GraphLevelGNN.load_from_checkpoint(pretrained_filename)
    else:
        print("未发现预训练模型，从零开始训练...")
        pl.seed_everything(42)
        # 【关键修改2】动态判断输出维度。如果是 2 分类（有毒/无毒），输出 1 个神经元就够了；多分类则输出类别数。
        c_out = 1 if dataset.num_classes == 2 else dataset.num_classes
        model = GraphLevelGNN(c_in=dataset.num_node_features, c_out=c_out, **model_kwargs)
        
        # 启动训练！注意这里传入的是 train 和 val 两个 loader
        trainer.fit(model, train_loader, val_loader)
        
        # 训练完读取巅峰权重
        model = GraphLevelGNN.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
    
    # 期末考试：分别在训练集和测试集上跑一遍
    train_result = trainer.test(model, train_loader, verbose=False)
    test_result = trainer.test(model, test_loader, verbose=False)
    
    result = {"test": test_result[0]['test_acc'], "train": train_result[0]['test_acc']} 
    return model, result

In [37]:
model, result = train_graph_classifier(model_name="GraphConv",
                                       dataset=tu_dataset,
                                        train_loader=graph_train_loader,
                                        val_loader=graph_val_loader,
                                        test_loader=graph_test_loader,                               
                                       c_hidden=256, 
                                       layer_name="GraphConv", 
                                       num_layers=3, 
                                       dp_rate_linear=0.5,
                                       dp_rate=0.0)

Seed set to 42
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Lightning automatically upgraded your loaded checkpoint from v1.0.2 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint d:\GNN\saved_models\tutorial7\GraphLevelGraphConv.ckpt`


发现预训练模型，正在光速加载...


d:\Anoconda\envs\gnn_env\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:485: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
d:\Anoconda\envs\gnn_env\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

d:\Anoconda\envs\gnn_env\lib\site-packages\pytorch_lightning\utilities\data.py:79: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 2. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Testing: |          | 0/? [00:00<?, ?it/s]

In [38]:
print(f"Train performance: {100.0*result['train']:4.2f}%")
print(f"Test performance:  {100.0*result['test']:4.2f}%")


Train performance: 93.28%
Test performance:  92.11%
